In [1]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from enum import IntEnum
from itertools import permutations
from typing import Any, Dict, Iterable, List, Sequence

In [5]:
# ------------------------------------------------------------
# 1) Action space
# ------------------------------------------------------------
class Action(IntEnum):
    STOP = 0
    RERANK = 1
    REFORMULATE = 2
    PRF = 3


HOT_INDEX = {
    Action.STOP: 0,
    Action.RERANK: 1,
    Action.REFORMULATE: 2,
    Action.PRF: 3,
}

In [ ]:
# ------------------------------------------------------------
# Reward configuration 
# ------------------------------------------------------------

# Weights for metric deltas
W_NDCG = 0.5
W_RECALL = 0.5

# Cost regularization strength
LAMBDA_COST = 0.1

# Per-action costs (tune to your infra / latency / $)
ACTION_COST = {
    Action.STOP: 0.0,
    Action.RERANK: 0.02,  
    Action.REFORMULATE: 0.10,  
    Action.PRF: 0.05,  
}

# Optional clipping for offline RL stability
REWARD_MIN = -1.0
REWARD_MAX = 1.0

In [ ]:
# ------------------------------------------------------------
# 2) Scenario definition
#    Assumption: the 10 scenarios are:
#    - stop immediately: 1
#    - one helper then stop: 3
#    - two distinct helpers then forced stop: 6
# ------------------------------------------------------------
@dataclass(frozen=True)
class Scenario:
    scenario_id: str
    helper_calls: tuple[Action, ...]

    @property
    def action_sequence(self) -> tuple[Action, ...]:
        # Always append STOP so every scenario terminates explicitly.
        return self.helper_calls + (Action.STOP,)


def generate_scenarios() -> List[Scenario]:
    helpers = (Action.RERANK, Action.REFORMULATE, Action.PRF)

    scenarios: List[Scenario] = [
        Scenario("stop_immediately", ())
    ]

    scenarios.extend(
        Scenario(f"{a.name.lower()}_then_stop", (a,))
        for a in helpers
    )

    scenarios.extend(
        Scenario(f"{a.name.lower()}_{b.name.lower()}_then_stop", (a, b))
        for a, b in permutations(helpers, 2)
        if a != Action.RERANK
    )

    assert len(scenarios) == 8
    return scenarios
    # ------------------------------------------------------------
# 3) Hot-vector helpers
#    Format: [stop, agent1, agent2, agent3]
# ------------------------------------------------------------

def zero_hot_vector() -> List[int]:
    return [0, 0, 0, 0]


def apply_action_to_hot_vector(
    current_hot: Sequence[int],
    action: Action,
) -> List[int]:
    next_hot = list(current_hot)
    next_hot[HOT_INDEX[action]] = 1
    return next_hot

def scenario_hot_trace(scenario: Scenario) -> List[List[int]]:
    """
    Example:
    Scenario(helper_calls=(AGENT2, AGENT1))
    -> [[0,0,1,0], [0,1,1,0], [1,1,1,0]]
    """
    hot = zero_hot_vector()
    trace: List[List[int]] = []

    for action in scenario.action_sequence:
        hot = apply_action_to_hot_vector(hot, action)
        trace.append(hot.copy())

    return trace

def compute_reward(
    state: Any,
    action: Action,
    next_state: Any,
    query_record: Dict[str, Any],
) -> float:
    """
    Compute r_t from (s_t, a_t, s_{t+1}).
    - `state` and `next_state` each contain the current retrieval metrics, e.g.:
        state["ndcg"]        : float  # nDCG@K before taking action
        state["recall"]      : float  # Recall@K before taking action
        next_state["ndcg"]   : float  # nDCG@K after taking action
        next_state["recall"] : float  # Recall@K after taking action
    - These metrics are computed using TREC relevance labels outside this function.
    """

    # 1) Metric improvements (can be negative)
    ndcg_prev = state.get("ndcg", 0.0)
    ndcg_curr = next_state.get("ndcg", 0.0)
    delta_ndcg = ndcg_curr - ndcg_prev

    recall_prev = state.get("recall", 0.0)
    recall_curr = next_state.get("recall", 0.0)
    delta_recall = recall_curr - recall_prev

    metric_reward = W_NDCG * delta_ndcg + W_RECALL * delta_recall

    # 2) Cost of the chosen action
    cost = ACTION_COST.get(action, 0.0)
    cost_penalty = LAMBDA_COST * cost

    # 3) Final reward r_t
    r_t = metric_reward - cost_penalty

    # 4) Optional clipping for offline RL stability
    if REWARD_MIN is not None and REWARD_MAX is not None:
        r_t = max(REWARD_MIN, min(REWARD_MAX, r_t))

    return r_t

In [14]:
scenarios = generate_scenarios()
for scenario in scenarios:
    print(f"{scenario.scenario_id}: {scenario.action_sequence}")

stop_immediately: (<Action.STOP: 0>,)
rerank_then_stop: (<Action.RERANK: 1>, <Action.STOP: 0>)
reformulate_then_stop: (<Action.REFORMULATE: 2>, <Action.STOP: 0>)
prf_then_stop: (<Action.PRF: 3>, <Action.STOP: 0>)
reformulate_rerank_then_stop: (<Action.REFORMULATE: 2>, <Action.RERANK: 1>, <Action.STOP: 0>)
reformulate_prf_then_stop: (<Action.REFORMULATE: 2>, <Action.PRF: 3>, <Action.STOP: 0>)
prf_rerank_then_stop: (<Action.PRF: 3>, <Action.RERANK: 1>, <Action.STOP: 0>)
prf_reformulate_then_stop: (<Action.PRF: 3>, <Action.REFORMULATE: 2>, <Action.STOP: 0>)


In [19]:
hot_vector = zero_hot_vector()
next_hot = apply_action_to_hot_vector(hot_vector, Action.RERANK)
print(f"Current hot vector: {hot_vector}")
print(f"Next hot vector: {next_hot}")
next_next_hot = apply_action_to_hot_vector(next_hot, Action.REFORMULATE)
print(f"Next next hot vector: {next_next_hot}")

Current hot vector: [0, 0, 0, 0]
Next hot vector: [0, 1, 0, 0]
Next next hot vector: [0, 1, 1, 0]


In [22]:
trace = scenario_hot_trace(scenarios[7])
print(f"Scenario: {scenarios[7].scenario_id}")
print(f"Hot trace: {trace}")

Scenario: prf_reformulate_then_stop
Hot trace: [[0, 0, 0, 1], [0, 0, 1, 1], [1, 0, 1, 1]]
